# Phase 1 — Prepare data

**Goal.** Turn a single large SMILES file (millions of molecules) into many small CSV "chunks" that can be processed independently on the GPU array in Phase 2.

**Why chunk at all?**

1. **Parallelism.** Each chunk becomes one Slurm array task on its own GPU. If you have 1,000 chunks and 16 concurrent tasks (the default), the screen finishes ~16× faster than a single-node run.
2. **Failure isolation.** If one chunk hits an OOM or a pathological molecule, only that chunk's CSV is lost — everything else keeps flowing.
3. **Checkpointing.** Partial results land on disk as chunks complete, so you can inspect progress without waiting for the full run.

**Where to run this notebook.** Head node, plain `python3.12` kernel. No GPU needed yet — this phase only reads and writes text files. Launch from the project root (`alchemi-demo/`).

## 1. What's in the shared data directory

The `alchemi-ht` shared container ships three GDB-17 subsets. GDB-17 is a database of **enumerated** small organic molecules up to 17 heavy atoms (~166 billion candidates total); here we have 50-million-molecule samples of it.

- `GDB17.50000000.smi.gz` — full 50M sample, unfiltered.
- `GDB17.50000000LL.smi.gz` — **L**ead-**L**ike subset (drug-like molecular weight and logP).
- `GDB17.50000000LLnoSR.smi.gz` — lead-like **with no small rings** (3- and 4-membered rings removed). This is the cleanest starting point for most screens and is the default in `1_prepare_data.py`.

Each file is a whitespace-separated `<SMILES>\t<id>` text file, gzip-compressed. Phase 1 streams line-by-line so memory use stays constant regardless of file size.

In [ ]:
!ls -lh /n/holylfs06/LABS/kempner_shared/Everyone/containers/applications/alchemi-ht/data/

## 2. Chunk a tiny slice for the smoke test

We'll take just **4 molecules** and split them into **2 chunks of 2** — the smallest possible end-to-end exercise.

`1_prepare_data.py` accepts:

| Flag | What it does | Default |
| --- | --- | --- |
| `--input` | Source `.smi(.gz)` file | shared GDB-17 LL-noSR |
| `--outdir` | Where to write chunk CSVs | `data/chunks` |
| `--chunk-size` | Molecules per chunk | `5000` |
| `--max-molecules` | Stop after N molecules (for smoke tests) | unlimited |

On stdout you should see one line like `molecules=4 chunks=2 outdir=data/chunks` — that chunk count is what Phase 2 will read back to size the Slurm array.

In [ ]:
!python3.12 tutorial/1_prepare_data.py --max-molecules 4 --chunk-size 2

## 3. What got written?

Expect exactly two CSV files. The filename pattern `chunk_{N}.csv` is how Phase 2 maps `${SLURM_ARRAY_TASK_ID}` to an input file.

In [ ]:
!ls data/chunks/

## 4. Inspect a chunk with pandas

Each chunk has two columns:

- `smiles` — the 1D chemical string, exactly as it appeared in the source file.
- `source_id` — a lightweight label (`<chunk>-<row>`) so you can trace a finished result back to its origin. Preserved end-to-end through Phase 3 and Phase 4 for debugging.

In [ ]:
import pandas as pd
df = pd.read_csv('data/chunks/chunk_1.csv')
df

## 5. Scaling up

For a real screen, drop `--max-molecules` (so the whole input file gets processed) and keep the default `--chunk-size 5000`:

```bash
python3.12 tutorial/1_prepare_data.py --chunk-size 5000
```

**Picking a chunk size.** Larger chunks = fewer Slurm jobs but longer wall-clock per task (risk of Slurm `--time` timeout). Smaller chunks = more scheduler overhead but better load balancing. 5,000 is a reasonable default for the B3LYP/def2-SVP configuration in Phase 3, which takes ~1–2 minutes per molecule on an H100. Adjust up for smaller basis sets, down for larger ones.

**Next:** open `02_launch_slurm.ipynb` to submit the array.